# Docling section-preserving DOCX POC

This notebook trims an inclusive, 1-based page range from a PDF, converts only those pages with Docling, creates an editable section-preserving DOCX with images and tables represented by dynamic tags and page headers, footers, footnotes, and edge-positioned page-number markers omitted, and chunks that DOCX by its Word heading hierarchy. It also saves the selected-page PDF, a Markdown layout diagnostic with raw PDF text positions and Docling-detected elements (including furniture for comparison), section index, chunk JSON, and an Excel workbook with `chunk_id` and `chunk_text`.

Set `PDF_PATH`, `START_PAGE`, and `END_PAGE` in the setup cell. Each PDF and selected page range gets its own folder under `outputs/section_docx_poc`.


In [34]:
from __future__ import annotations

from collections import Counter
from io import BytesIO
from pathlib import Path
import json
import sys

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docx import Document
from docx.shared import Inches, Pt
from pypdf import PdfReader, PdfWriter

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
PDF_PATH = Path("/Users/suyashutekar/Projects/pdflens_replacement/rbi_docs/51916.pdf")
START_PAGE = 1  # 1-based, inclusive
END_PAGE = 10    # 1-based, inclusive; set to None to use the final PDF page

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}. Set PDF_PATH to your PDF.")

source_reader = PdfReader(str(PDF_PATH))
SOURCE_PAGE_COUNT = len(source_reader.pages)
if END_PAGE is None:
    END_PAGE = SOURCE_PAGE_COUNT
if not (1 <= START_PAGE <= END_PAGE <= SOURCE_PAGE_COUNT):
    raise ValueError(
        f"Invalid page range {START_PAGE}-{END_PAGE}; PDF has {SOURCE_PAGE_COUNT} pages. "
        "Page numbers are 1-based and inclusive."
    )

PAGE_RANGE_LABEL = f"pages_{START_PAGE}-{END_PAGE}"
OUTPUT_DIR = (
    PROJECT_ROOT / "outputs" / "section_docx_poc"
    / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRIMMED_PDF_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}.pdf"
DOCX_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_sections.docx"
SECTION_INDEX_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_sections.json"
MARKDOWN_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_layout.md"
CHUNK_JSON_PATH = OUTPUT_DIR / f"{DOCX_PATH.stem}_chunks.json"
CHUNK_XLSX_PATH = OUTPUT_DIR / f"{DOCX_PATH.stem}_chunks.xlsx"

writer = PdfWriter()
for source_index in range(START_PAGE - 1, END_PAGE):
    writer.add_page(source_reader.pages[source_index])
with TRIMMED_PDF_PATH.open("wb") as trimmed_pdf_file:
    writer.write(trimmed_pdf_file)

print(f"Input PDF: {PDF_PATH}")
print(f"Selected source pages: {START_PAGE}-{END_PAGE} of {SOURCE_PAGE_COUNT}")
print(f"Trimmed PDF: {TRIMMED_PDF_PATH}")
print(f"DOCX: {DOCX_PATH}")


Input PDF: /Users/suyashutekar/Projects/pdflens_replacement/rbi_docs/51916.pdf
Selected source pages: 1-10 of 45
Trimmed PDF: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10.pdf
DOCX: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_sections.docx


## 1. Convert the selected pages with Docling

`START_PAGE` and `END_PAGE` are 1-based and inclusive. The selected pages are first copied into a separate PDF; Docling, furniture extraction, DOCX generation, and chunking then use that trimmed PDF/DOCX.


In [35]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.generate_parsed_pages = True
pipeline_options.heading_hierarchy_options.enabled = True
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)
conversion = converter.convert(str(TRIMMED_PDF_PATH))
doc = conversion.document

print(f"Selected pages processed: {doc.num_pages()} (source pages {START_PAGE}-{END_PAGE})")
print(f"Text items: {len(doc.texts)} | Tables: {len(doc.tables)} | Pictures: {len(doc.pictures)}")


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 13734.74it/s]


Selected pages processed: 10 (source pages 1-10)
Text items: 160 | Tables: 0 | Pictures: 1


## 2. Inspect detected sections

Review Docling's heading labels, levels, and page provenance before generating the DOCX.


In [36]:
def label_name(item):
    label = getattr(item, "label", "")
    return str(getattr(label, "name", getattr(label, "value", label))).upper()

def first_page_number(item):
    provenance = getattr(item, "prov", None) or []
    return getattr(provenance[0], "page_no", None) if provenance else None

heading_preview = []
for item, tree_level in doc.iterate_items():
    if label_name(item) in {"SECTION_HEADER", "TITLE"}:
        heading_preview.append({
            "page": (first_page_number(item) + START_PAGE - 1)
                    if first_page_number(item) is not None else None,
            "level": getattr(item, "level", tree_level or 1),
            "label": label_name(item),
            "text": getattr(item, "text", ""),
            "top": float(getattr((getattr(item, "prov", None) or [None])[0], "bbox", None).t)
                   if (getattr(item, "prov", None) or []) else 0,
        })

heading_preview.sort(key=lambda heading: (heading["page"] or 0, -heading["top"]))
print(f"Detected {len(heading_preview)} title/section headings")
for heading in heading_preview[:80]:
    level = max(1, min(int(heading["level"] or 1), 9))
    print(f"source p.{heading['page']} | {'  ' * (level - 1)}{heading['text']}")


Detected 39 title/section headings
source p.2 |     - Priority Sector Lending -
source p.2 | Prime Minister's Rozgar Yojana (PMRY)
source p.2 |           Master Circular PRIME MINISTER'S ROZGAR YOJANA (PMRY) Contents
source p.4 |           Master Circular
source p.4 |           PRIORITY SECTOR LENDINGS - SPECIAL PROGRAMMES
source p.4 | Section - I
source p.4 | Guidelines on Prime Minister's Rozgar Yojana (PMRY)
source p.4 |   1. Objective
source p.4 |   2. Coverage
source p.4 |   3. Target Group
source p.4 |   4. Eligibility Norms
source p.4 |     (i) Age
source p.4 |     (ii) Education
source p.4 |     (iii) Annual family income
source p.5 |     iv). Residency
source p.5 |     (v) Other conditions
source p.5 |   5. Eligible Activities
source p.6 |   6. Relaxations of PMRY Norms for North-East Region, Himachal Pradesh, Uttaranchal and Jammu & Kashmir
source p.6 |   7. Project Funding
source p.6 |     (i) Project preparation
source p.6 |     (ii) Components of project cost
source p.6 | 

## 2. Build the section-preserving DOCX for the selected pages

Docling content is grouped by trimmed-page order and written into separate Word sections. Images and tables become dynamic `<doc_ai>` tags; headers, footers, footnotes, and short page-number markers near any page edge are omitted from the DOCX (they remain visible in the Markdown diagnostic). The section index records original source-page numbers using the selected-range offset.


### Page-number removal update

The implementation is in the **DOCX builder code cell** (the code cell immediately after “Build the section-preserving DOCX”). It removes page-number-shaped text only when it is near a page edge and matches a standalone candidate in the PDF text layer at the same location. This protects numeric values split from index or table rows. See the inline comments `PAGE-NUMBER FILTER`, `Remove page-number candidates`, and `Omit Docling page-number elements`.


In [37]:
from collections import defaultdict
import re
import unicodedata

import pdfplumber
from docx.enum.section import WD_SECTION
from docx.shared import Inches, Pt, RGBColor
from docx.oxml.ns import qn

word = Document()
word.core_properties.title = PDF_PATH.stem
word.core_properties.subject = "Docling section-preservation proof of concept"

first_page = next(iter(doc.pages.values()), None)
if first_page is not None and getattr(first_page, "size", None) is not None:
    section = word.sections[0]
    section.page_width = Pt(float(first_page.size.width))
    section.page_height = Pt(float(first_page.size.height))
    section.top_margin = Inches(0.55)
    section.bottom_margin = Inches(0.55)
    section.left_margin = Inches(0.58)
    section.right_margin = Inches(0.58)

# Compact, readable typography helps each source page fit on its corresponding DOCX page.
normal = word.styles["Normal"]
normal.font.name = "Arial"
normal.font.size = Pt(9)
normal.paragraph_format.space_after = Pt(3)
normal.paragraph_format.line_spacing = 1.0
title_style = word.styles["Title"]
title_style.font.color.rgb = RGBColor(0, 0, 0)
title_style.font.underline = False
title_ppr = title_style._element.get_or_add_pPr()
title_border = title_ppr.find(qn("w:pBdr"))
if title_border is not None:
    title_ppr.remove(title_border)

for style_name, size in (("Title", 18), ("Heading 1", 14), ("Heading 2", 12),
                         ("Heading 3", 10.5), ("Heading 4", 9.5), ("Heading 5", 9),
                         ("Heading 6", 9), ("Heading 7", 9), ("Heading 8", 9),
                         ("Heading 9", 9)):
    style = word.styles[style_name]
    style.font.name = "Arial"
    style.font.size = Pt(size)
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(2)
    style.paragraph_format.keep_with_next = True


def normalize(value):
    value = unicodedata.normalize("NFKC", str(value or "")).casefold()
    value = re.sub(r"\(?cid:\d+\)?", " ", value)
    value = re.sub(r"^\s*\d+[.)]\s+", " ", value)
    return re.sub(r"[^\w]+", " ", value).strip()


def clean_pdf_text(value):
    return re.sub(r"\(?cid:\d+\)?", "•", str(value or "")).strip()


# PAGE-NUMBER FILTER: recognize standalone labels, then require edge-position geometry.
def is_page_number_text(value):
    """Recognize short, standalone printed page-number forms, not body sentences."""
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    return bool(re.fullmatch(
        r"(?:(?:page|p)\.?\s*)?\d{1,4}(?:\s*(?:/|of)\s*\d{1,4})?",
        text,
        flags=re.IGNORECASE,
    ))


def is_near_page_edge(left, right, top, bottom, page_width, page_height):
    """Page-edge zones scale with the actual page size, including rotated/sized pages."""
    horizontal_edge = page_width * 0.08
    vertical_edge = page_height * 0.08
    return (
        left <= horizontal_edge
        or right >= page_width - horizontal_edge
        or top <= vertical_edge
        or bottom >= page_height - vertical_edge
    )


def is_page_number_line(line, page_width, page_height):
    if not is_page_number_text(line["text"]):
        return False
    words = line["words"]
    left = min(word["x0"] for word in words)
    right = max(word["x1"] for word in words)
    top = min(word["top"] for word in words)
    bottom = max(word["bottom"] for word in words)
    return is_near_page_edge(left, right, top, bottom, page_width, page_height)


def is_page_number_item(text, bbox, page_data):
    """Match a Docling number to a standalone PDF marker at the same location.

    This prevents an index/table value (for example, the trailing 40 in an index
    row) from being removed merely because Docling split it into a numeric item.
    """
    if not is_page_number_text(text) or bbox is None:
        return False

    # Docling bounding boxes use a bottom-left origin; compare in top-origin points.
    item_left = float(bbox.l)
    item_right = float(bbox.r)
    item_top = page_data["height"] - float(bbox.t)
    item_bottom = page_data["height"] - float(bbox.b)
    if not is_near_page_edge(
        item_left, item_right, item_top, item_bottom,
        page_data["width"], page_data["height"],
    ):
        return False

    item_text = normalize(text)
    tolerance_x = max(6.0, page_data["width"] * 0.012)
    tolerance_y = max(6.0, page_data["height"] * 0.012)
    for line in page_data["page_number_lines"]:
        if normalize(line["text"]) != item_text:
            continue
        words = line["words"]
        line_left = min(word["x0"] for word in words)
        line_right = max(word["x1"] for word in words)
        line_top = min(word["top"] for word in words)
        line_bottom = max(word["bottom"] for word in words)
        same_source_location = (
            abs(item_left - line_left) <= tolerance_x
            and abs(item_right - line_right) <= tolerance_x
            and abs(item_top - line_top) <= tolerance_y
            and abs(item_bottom - line_bottom) <= tolerance_y
        )
        if same_source_location:
            return True
    return False


def pdf_page_lines(page):
    """Group PDF words into visual lines while retaining top-origin coordinates."""
    words = sorted(page.extract_words(), key=lambda item: (item["top"], item["x0"]))
    lines = []
    for word_item in words:
        match = next((line for line in reversed(lines)
                      if abs(line["top"] - word_item["top"]) <= 2.5), None)
        if match is None:
            match = {"top": word_item["top"], "words": []}
            lines.append(match)
        match["words"].append(word_item)
    for line in lines:
        line["words"].sort(key=lambda item: item["x0"])
        line["text"] = " ".join(item["text"] for item in line["words"])
    return sorted(lines, key=lambda line: line["top"])


pdf_page_data = []
with pdfplumber.open(TRIMMED_PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        page_lines = pdf_page_lines(page)
        top_band = page.height * 0.055
        bottom_band = page.height * 0.055
        header_lines = [line for line in page_lines if line["top"] <= top_band]
        footer_lines = [line for line in page_lines
                        if line["top"] >= page.height - bottom_band]
        # Remove page-number candidates from PDF fallback body text at any page edge.
        page_number_lines = [
            line for line in page_lines
            if is_page_number_line(line, page.width, page.height)
        ]
        excluded_from_body = header_lines + footer_lines + page_number_lines
        pdf_page_data.append({
            "width": page.width,
            "height": page.height,
            "header": "   |   ".join(clean_pdf_text(line["text"]) for line in header_lines),
            "footer": "   |   ".join(clean_pdf_text(line["text"]) for line in footer_lines),
            "page_number_lines": page_number_lines,
            "furniture_text": " ".join(
                line["text"] for line in header_lines + footer_lines + page_number_lines
            ),
            "lines": page_lines,
            "body_lines": [line for line in page_lines if line not in excluded_from_body],
        })

markdown_parts = [
    f"# {PDF_PATH.stem} layout and section diagnostics",
    f"Source PDF: `{PDF_PATH.name}`  \
Selected source pages: {START_PAGE}-{END_PAGE}",
    "This diagnostic compares selectable PDF text and coordinates with Docling's detected elements and Markdown conversion. Source coordinates are measured in PDF points from the page's top-left. Lines are grouped by approximate text baseline.",
    "## Page headers and footers",
]
for selected_page, page_data in enumerate(pdf_page_data, start=1):
    source_page = selected_page + START_PAGE - 1
    markdown_parts.append(f"### Source page {source_page}")
    markdown_parts.append(f"**Header:** {page_data['header'] or '(none detected)'}")
    markdown_parts.append(f"**Footer:** {page_data['footer'] or '(none detected)'}")

markdown_parts.extend([
    "## Page-number candidates omitted from the DOCX",
    "Short standalone numeric page labels (for example, `2`, `2/42`, or `Page 2`) are omitted when their text box falls within a size-relative edge zone on any side of the page. The raw source lines remain in the table below for diagnosis.",
])
for selected_page, page_data in enumerate(pdf_page_data, start=1):
    source_page = selected_page + START_PAGE - 1
    candidates = page_data["page_number_lines"]
    labels = [clean_pdf_text(line["text"]) for line in candidates]
    markdown_parts.append(
        f"- Source page {source_page}: " + (", ".join(f"`{label}`" for label in labels) if labels else "(none detected)")
    )

markdown_parts.extend([
    "## Source PDF text lines",
    "These rows come directly from the PDF text layer. Compare their coordinates and wording with the Docling elements below to see whether sections were already on the same visual line in the source.",
    "| Source page | Top (pt) | Left (pt) | Right (pt) | Source text |",
    "| ---: | ---: | ---: | ---: | --- |",
])
for selected_page, page_data in enumerate(pdf_page_data, start=1):
    source_page = selected_page + START_PAGE - 1
    for line in page_data["lines"]:
        left = min(word["x0"] for word in line["words"])
        right = max(word["x1"] for word in line["words"])
        source_text = clean_pdf_text(line["text"]).replace("|", "\\|")
        markdown_parts.append(
            f"| {source_page} | {line['top']:.1f} | {left:.1f} | {right:.1f} | {source_text} |"
        )

markdown_parts.extend([
    "## Docling detected elements",
    "This list shows Docling's labels, heading levels, and page coordinates before DOCX formatting.",
    "| Source page | Label | Level | Top (pt) | Left (pt) | Detected text/content |",
    "| ---: | --- | ---: | ---: | ---: | --- |",
])
for item, tree_level in doc.iterate_items():
    local_page = first_page_number(item)
    if local_page is None:
        continue
    source_page = local_page + START_PAGE - 1
    provenance = (getattr(item, "prov", None) or [None])[0]
    bbox = getattr(provenance, "bbox", None) if provenance else None
    page_height = pdf_page_data[local_page - 1]["height"]
    top = page_height - float(bbox.t) if bbox is not None else ""
    left = float(bbox.l) if bbox is not None else ""
    label = label_name(item)
    level = getattr(item, "level", tree_level or "") if label in {"SECTION_HEADER", "TITLE"} else ""
    detected_text = str(getattr(item, "text", "") or "").strip()
    if label in {"TABLE", "TABLE_ITEM"}:
        detected_text = item.export_to_markdown(doc=doc)
    elif label in {"PICTURE", "PICTURE_ITEM"} and not detected_text:
        detected_text = "[picture element]"
    detected_text = re.sub(r"\s+", " ", detected_text).replace("|", "\\|")
    markdown_parts.append(
        f"| {source_page} | {label} | {level} | {top if top == '' else f'{top:.1f}'} | "
        f"{left if left == '' else f'{left:.1f}'} | {detected_text} |"
    )

markdown_parts.extend(["## Docling Markdown conversion", doc.export_to_markdown()])
MARKDOWN_PATH.write_text("\n\n".join(markdown_parts).rstrip() + "\n", encoding="utf-8")
print(f"Saved Markdown layout diagnostics: {MARKDOWN_PATH}")

items_by_page = defaultdict(list)
recognized_text_by_page = defaultdict(list)
for item, tree_level in doc.iterate_items():
    page_no = first_page_number(item)
    if page_no is None:
        continue
    provenance = (getattr(item, "prov", None) or [None])[0]
    bbox = getattr(provenance, "bbox", None) if provenance else None
    # Docling coordinates are bottom-origin; negative top sorts topmost first.
    top = float(getattr(bbox, "t", 0)) if bbox is not None else 0.0
    left = float(getattr(bbox, "l", 0)) if bbox is not None else 0.0
    items_by_page[page_no].append((top, left, item, tree_level))
    text = str(getattr(item, "text", "") or "").strip()
    if text:
        recognized_text_by_page[page_no].append(text)
    if label_name(item) in {"TABLE", "TABLE_ITEM"}:
        recognized_text_by_page[page_no].append(item.export_to_markdown(doc=doc))

# Reconcile the PDF's selectable text with Docling's output. Add any complete
# source text line that Docling did not emit, preserving its page and position.
for page_number, page_data in enumerate(pdf_page_data, start=1):
    recognized_text = normalize(" ".join(recognized_text_by_page[page_number]))
    structured_regions = []
    for item, _tree_level in doc.iterate_items():
        if label_name(item) not in {"TABLE", "TABLE_ITEM", "PICTURE", "PICTURE_ITEM"}:
            continue
        provenance = (getattr(item, "prov", None) or [None])[0]
        if provenance is None or provenance.page_no != page_number:
            continue
        bbox = getattr(provenance, "bbox", None)
        if bbox is not None:
            structured_regions.append(bbox)
    for line in page_data["body_lines"]:
        line_text = clean_pdf_text(line["text"])
        normalized_line = normalize(line_text)
        if not normalized_line or normalized_line in recognized_text:
            continue
        line_left = min(word["x0"] for word in line["words"])
        line_right = max(word["x1"] for word in line["words"])
        line_top = page_data["height"] - line["top"]
        line_bottom = page_data["height"] - max(word["bottom"] for word in line["words"])
        already_in_structured_region = any(
            line_right >= float(bbox.l) - 4 and line_left <= float(bbox.r) + 4
            and line_top >= float(bbox.b) - 28 and line_bottom <= float(bbox.t) + 28
            for bbox in structured_regions
        )
        if not already_in_structured_region:
            items_by_page[page_number].append(
                (line_top, line_left,
                 {"label": "PDF_FALLBACK", "text": line_text}, 1)
            )

section_index = []
block_counts = Counter()
table_id = 0
image_id = 0


def record_block(category):
    block_counts[category] += 1


for page_number in range(1, doc.num_pages() + 1):
    if page_number > 1:
        word.add_section(WD_SECTION.NEW_PAGE)
    page_items = sorted(items_by_page.get(page_number, []), key=lambda entry: (-entry[0], entry[1]))
    for _top, _left, item, tree_level in page_items:
        label = ("PDF_FALLBACK" if isinstance(item, dict) and item.get("label") == "PDF_FALLBACK"
                 else label_name(item))
        text = str(item.get("text", "") if isinstance(item, dict)
                   else getattr(item, "text", "") or "").strip()
        if label in {"PAGE_HEADER", "PAGE_FOOTER", "FOOTNOTE", "FOOTNOTE_REFERENCE"}:
            record_block("furniture_or_footnote_skipped")
            continue

        # Omit Docling page-number elements from the generated DOCX as well.
        page_data = pdf_page_data[page_number - 1]
        item_provenance = (getattr(item, "prov", None) or [None])[0] if not isinstance(item, dict) else None
        item_bbox = getattr(item_provenance, "bbox", None) if item_provenance else None
        if is_page_number_item(text, item_bbox, page_data):
            record_block("page_number_skipped")
            continue

        if (text and label not in {"SECTION_HEADER", "TITLE"}
                and normalize(text) in normalize(page_data["furniture_text"])):
            continue

        if label in {"SECTION_HEADER", "TITLE"} and text:
            # Place the document's detected title at the start of its page instead of
            # relying on Docling's tree order, which can yield it after body content.
            level = max(1, min(int(getattr(item, "level", tree_level or 1) or 1), 9))
            if page_number == 1 and level == 1 and not section_index:
                word.add_paragraph(text, style="Title")
                stored_level = 1
            else:
                word.add_heading(text, level=min(level, 9))
                stored_level = level
            source_page_number = page_number + START_PAGE - 1
            section_index.append({"title": text, "level": stored_level,
                                  "start_page": source_page_number,
                                  "end_page": source_page_number})
            record_block("heading")
            continue

        if label in {"TABLE", "TABLE_ITEM"}:
            table_id += 1
            word.add_paragraph(f'<doc_ai type="table" id="table_{table_id}"/>')
            record_block("table_tag")
            continue
        if label in {"PICTURE", "PICTURE_ITEM"}:
            image_id += 1
            word.add_paragraph(f'<doc_ai type="image" id="image_{image_id}"/>')
            record_block("image_tag")
            continue
        if not text:
            continue
        if label in {"LIST_ITEM", "ORDERED_LIST", "UNORDERED_LIST"}:
            style = "List Number" if label == "ORDERED_LIST" else "List Bullet"
            paragraph = word.add_paragraph(text, style=style)
            record_block("list_item")
        else:
            word.add_paragraph(text)
            record_block("pdf_text_fallback" if label == "PDF_FALLBACK" else "text")

word.save(DOCX_PATH)
SECTION_INDEX_PATH.write_text(json.dumps(section_index, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved: {DOCX_PATH}")
print(f"Saved: {SECTION_INDEX_PATH}")
print(f"Content blocks: {dict(block_counts)}")
print(f"Section headings: {len(section_index)}")


Saved Markdown layout diagnostics: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_layout.md
Saved: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_sections.docx
Saved: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_sections.json
Content blocks: {'image_tag': 1, 'text': 35, 'heading': 39, 'list_item': 69, 'pdf_text_fallback': 14}
Section headings: 39


## 3. Chunk the selected-range DOCX by section

The chunker reads the generated DOCX, prefixes each chunk with its heading path using ` | ` separators, and saves both JSON and an Excel workbook.


In [38]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from chunking_strategy.sectionwise_chunking import chunk_docx, save_chunks_xlsx

check = Document(DOCX_PATH)
heading_counts = Counter(
    paragraph.style.name for paragraph in check.paragraphs
    if paragraph.style.name.startswith("Heading ")
)
print(f"DOCX size: {DOCX_PATH.stat().st_size:,} bytes")
print(f"Word heading styles: {dict(heading_counts)}")
print(f"Native Word tables: {len(check.tables)} (expected 0; tables are represented by tags)")
print(f"Image tags: {sum(1 for paragraph in check.paragraphs if '<doc_ai type="image"' in paragraph.text)}")
print(f"Table tags: {sum(1 for paragraph in check.paragraphs if '<doc_ai type="table"' in paragraph.text)}")
print(f"Indexed sections: {len(json.loads(SECTION_INDEX_PATH.read_text(encoding='utf-8')))}")

if not section_index:
    print("WARNING: Docling detected no section headings. The output cannot preserve a hierarchy it did not identify.")
elif not check.paragraphs:
    raise RuntimeError("Generated DOCX has no paragraphs.")

chunk_result = chunk_docx(DOCX_PATH)
CHUNK_JSON_PATH.write_text(
    json.dumps(chunk_result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
save_chunks_xlsx(chunk_result["chunks"], CHUNK_XLSX_PATH)
print(f"Section chunks: {chunk_result['chunk_count']}")
print(f"Chunk JSON: {CHUNK_JSON_PATH}")
print(f"Chunk Excel: {CHUNK_XLSX_PATH}")


DOCX size: 45,935 bytes
Word heading styles: {'Heading 3': 24, 'Heading 1': 3, 'Heading 6': 3, 'Heading 2': 9}
Native Word tables: 0 (expected 0; tables are represented by tags)
Image tags: 1
Table tags: 0
Indexed sections: 39
Section chunks: 32
Chunk JSON: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_sections_chunks.json
Chunk Excel: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/51916_pages_1-10/51916_pages_1-10_sections_chunks.xlsx


## Notes

- Page ranges are 1-based and inclusive. Set `END_PAGE = None` to include every page from `START_PAGE` through the end of the PDF.
- The trimmed PDF and a Markdown layout preview are saved as separate outputs; the original PDF is not modified. The Markdown compares source PDF text lines and coordinates with Docling labels, heading levels, furniture, and Markdown conversion; it is intended to diagnose whether a merge originated in source layout or extraction.
- The section index and DOCX headings use source PDF page numbers. The chunker groups content using the Word heading hierarchy and writes the heading path plus body text separated by ` | `.
- DOCX pagination remains editable/reflowable within the selected page range. Dense source pages may flow differently in Word.
